In [2]:
import cv2
import numpy as np
import math
from ultralytics import YOLO

def get_center(box):
    x1, y1, x2, y2 = box
    return int((x1 + x2) / 2), int((y1 + y2) / 2)

def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def smooth(data, window=5):
    if len(data) < window:
        return data
    return np.convolve(data, np.ones(window)/window, mode='valid')

VIDEO_PATH = "video1.mp4"
MODEL_PATH = "yolov8n.pt"   
TARGET_CLASS = None

MOVEMENT_THRESHOLD = 2      # pixels
STOP_WINDOW = 5             # frames
SMOOTH_WINDOW = 5

model = YOLO(MODEL_PATH)

# =========================
# HELPER FUNCTIONS
# =========================


# =========================
# VIDEO SETUP
# =========================
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)

positions = []
times = []
frame_count = 0

# =========================
# MAIN LOOP
# =========================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)

    detected_center = None

    for r in results:
        for box, cls in zip(r.boxes.xyxy, r.boxes.cls):
            class_name = model.names[int(cls)]

            if TARGET_CLASS is None or class_name == TARGET_CLASS:
                x1, y1, x2, y2 = map(int, box)
                detected_center = get_center((x1, y1, x2, y2))

                # Draw detection (optional)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
                cv2.circle(frame, detected_center, 5, (0,0,255), -1)
                break
        if detected_center:
            break

    if detected_center:
        positions.append(detected_center)
        times.append(frame_count / fps)

    frame_count += 1

    # OPTIONAL: display
    cv2.imshow("Tracking", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

# =========================
# POST PROCESSING
# =========================
if len(positions) < 10:
    print("Not enough detections.")
    exit()

# Separate X and Y
x_vals = [p[0] for p in positions]
y_vals = [p[1] for p in positions]

# Smooth Y to reduce noise
y_smooth = smooth(y_vals, SMOOTH_WINDOW)
t_smooth = times[:len(y_smooth)]

# =========================
# DETECT TOUCH GROUND
# =========================
touch_time = None

for i in range(1, len(y_smooth)):
    prev_y = y_smooth[i-1]
    curr_y = y_smooth[i]

    # Falling → stop falling (hit ground)
    if curr_y >= prev_y:
        touch_time = t_smooth[i]
        break

# =========================
# DETECT STOP MOVING
# =========================
stop_time = None

for i in range(STOP_WINDOW, len(positions)):
    movement = 0
    for j in range(i - STOP_WINDOW, i):
        movement += distance(positions[j], positions[j+1])

    if movement < MOVEMENT_THRESHOLD:
        stop_time = times[i]
        break

# =========================
# RESULTS
# =========================
print("/n===== RESULTS =====")
print("Touch ground at:", touch_time, "seconds")
print("Stop moving at:", stop_time, "seconds")

/n===== RESULTS =====
Touch ground at: 1.1666666666666667 seconds
Stop moving at: None seconds


In [2]:
import cv2
import numpy as np
import math

VIDEO_PATH = "video.mp4"

# =========================
# FUNCTION: calculate angle
# =========================
def calculate_angle_from_line(vx, vy):
    # angle between line and vertical
    angle_rad = math.atan2(vx, vy)
    angle_deg = abs(math.degrees(angle_rad))
    return angle_deg

# =========================
# VIDEO
# =========================
cap = cv2.VideoCapture(VIDEO_PATH)

angles = []
times = []

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # blur → reduce noise
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # edge detection
    edges = cv2.Canny(blur, 50, 150)

    # find contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        # largest contour = paper
        c = max(contours, key=cv2.contourArea)

        if cv2.contourArea(c) > 1000:
            # fit a line to contour
            [vx, vy, x, y] = cv2.fitLine(c, cv2.DIST_L2, 0, 0.01, 0.01)

            angle = calculate_angle_from_line(vx, vy)

            angles.append(angle)
            times.append(frame_count / fps)

            # draw line
            h, w = frame.shape[:2]
            lefty = int((-x * vy / vx) + y)
            righty = int(((w - x) * vy / vx) + y)

            cv2.line(frame, (w-1, righty), (0, lefty), (0, 255, 0), 2)

            # show angle
            cv2.putText(frame, f"Angle: {angle:.2f}",
                        (30, 30),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1, (0, 0, 255), 2)

    frame_count += 1

    cv2.imshow("Paper Bend Detection", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

# =========================
# RESULTS
# =========================
print("\n===== RESULTS =====")
print("Max bend angle:", max(angles) if angles else None)


===== RESULTS =====
Max bend angle: None


In [ ]:
import librosa
import numpy as np
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

AUDIO_PATH = "breathing.mp3"

# =========================
# LOAD AUDIO
# =========================
y, sr = librosa.load(AUDIO_PATH, sr=None)

# =========================
# AMPLITUDE ENVELOPE
# =========================
frame_size = 1024
hop_length = 512

amplitude_envelope = []

for i in range(0, len(y), hop_length):
    frame = y[i:i+frame_size]
    amplitude_envelope.append(max(frame))

amplitude_envelope = np.array(amplitude_envelope)

# =========================
# SMOOTH SIGNAL
# =========================
window_size = 10
smooth = np.convolve(amplitude_envelope,
                     np.ones(window_size)/window_size,
                     mode='same')

# =========================
# PEAK DETECTION
# =========================
peaks, _ = find_peaks(
    smooth,
    distance=sr / hop_length * 1.5,  # minimum time between breaths (~1.5 sec)
    prominence=0.02
)

# =========================
# BREATH PER MINUTE
# =========================
duration_sec = len(y) / sr
breath_count = len(peaks)

bpm = (breath_count / duration_sec) * 60

print("\n===== RESULTS =====")
print("Breath count:", breath_count)
print("Breaths per minute (BPM):", bpm)

# =========================
# OPTIONAL: PLOT
# =========================
plt.plot(smooth)
plt.plot(peaks, smooth[peaks], "rx")
plt.title("Breathing Detection")
plt.show()